# AI Power Portfolio — Colab

Put your repo URL below and run all the cells in order (Runtime > Run all).

In [ ]:
REPO_URL = "https://github.com/r0988137/CapstoneAI.git"


In [ ]:
import subprocess, os

repo_name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
subprocess.run(f"rm -rf {repo_name}", shell=True)
r = subprocess.run(f"timeout 120 git clone --depth 1 --progress {REPO_URL}", shell=True)
assert r.returncode == 0, "The clone failed or took longer than 2 minutes. Check that the repo is public and has no heavy files."

# find every folder that looks like the project (src/ + requirements.txt), at any depth
candidates = []
for dirpath, dirnames, filenames in os.walk(repo_name):
    if ".git" in dirpath:
        continue
    if "src" in dirnames and "requirements.txt" in filenames and os.path.isdir(os.path.join(dirpath, "src", "workflow")):
        candidates.append(dirpath)
print("Project folders found:", candidates)

# prefer the one that has the newest app (src/app/app.py); the repo may also contain an older copy
complete = [d for d in candidates if os.path.exists(os.path.join(d, "src", "app", "app.py"))]
assert candidates, "src/workflow not found in the repo. New files are missing on GitHub (did you upload the .zip without unzipping it?)"
assert complete, "Found the project, but src/app/app.py is missing in: " + str(candidates) + ". Upload the src/app folder to GitHub."
project_root = complete[0]
os.chdir(project_root)
print("Project folder:", os.getcwd())

!pip install -q -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/ai_power_portfolio_data"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ["WAREHOUSE_DB_PATH"] = f"{DRIVE_DIR}/warehouse.db"


In [ ]:
!python -m src.db.init_db
!python -m src.etl.load_market_data


In [ ]:
import sqlite3, pandas as pd

conn = sqlite3.connect(os.environ["WAREHOUSE_DB_PATH"])
df = pd.read_sql("""
    SELECT a.ticker, d.full_date, f.close
    FROM fact_market_price f
    JOIN dim_asset a ON a.asset_key = f.asset_key
    JOIN dim_date d ON d.date_key = f.date_key
    ORDER BY d.full_date DESC
    LIMIT 20
""", conn)
conn.close()
df


In [ ]:
!python -m src.etl.validate_market_data


## Questionnaire + full workflow (Weeks 5–10)

Optional: save your `ANTHROPIC_API_KEY` in Colab (key icon > Secrets). If you skip it, everything still works with the built-in text template. The questionnaire is answered only once, in the last cell below.

In [ ]:
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded: the LLM will be used")
except Exception:
    print("No API key: the built-in template will be used (everything works the same)")

In [ ]:
from src.questionnaire.collect import collect_answers_interactive
from src.workflow.run_workflow import run_full_workflow, format_report

answers = collect_answers_interactive()
conn = sqlite3.connect(os.environ["WAREHOUSE_DB_PATH"])
result = run_full_workflow(conn, "demo-user", answers)   # use include_news=False to skip the news step
conn.close()
print(format_report(result))

In [ ]:
conn = sqlite3.connect(os.environ["WAREHOUSE_DB_PATH"])
key = result["user_profile_key"]
pd.read_sql("""
    SELECT s.scenario_name, f.horizon_year, f.projected_value
    FROM fact_forecast f JOIN dim_scenario s ON s.scenario_key = f.scenario_key
    WHERE f.user_profile_key = ? ORDER BY f.horizon_year, s.scenario_name
""", conn, params=(key,)).pivot(index="horizon_year", columns="scenario_name", values="projected_value").plot(title="Forecast")

## Weeks 11–12: web app + dashboard

This cell opens the app (form, results and dashboard). Look for the link that says **Running on public URL** and open it. If you have already loaded data (ETL), the dashboard shows it.

In [ ]:
import sys, os

# 1) Check that the app files are really in the repo that Colab downloaded.
needed = ["src/app/app.py", "src/app/views.py", "src/app/__init__.py", "src/config/settings.py"]
missing = [f for f in needed if not os.path.exists(f)]
print("Working folder:", os.getcwd())
if missing:
    print("MISSING FILES in the downloaded repo:", missing)
    print("Folders found in src/:", sorted(os.listdir("src")) if os.path.isdir("src") else "no src folder")
    print("-> On GitHub, open your repo and check that these files exist (src/app/ is a NEW folder in v6 and later).")
    print("-> If they are missing, upload the src/app folder again (drag the whole folder, or use Add file > Upload files).")
    print("-> Then Runtime > Disconnect and delete runtime, and run all cells again.")
else:
    # 2) Make sure the newest code is used (Colab keeps old modules in memory between runs).
    for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
        del sys.modules[name]
    try:
        from src.app.app import launch
        launch(share=True)   # look for the "Running on public URL" link below
    except Exception as e:
        print("The app could not start:", type(e).__name__, "-", e)
        print("If the error mentions a missing name or module, use Runtime > Disconnect and delete runtime, then run all cells again.")
        print("If it mentions the share link, just run this cell once more (the public tunnel sometimes fails the first time).")
        raise